# Protein Interaction V1 Sample Test Notebook

This notebook runs a tiny end-to-end smoke test for the new protein-level interaction pipeline:
1. Build tiny pair index splits
2. Sanity-check one batched sample
3. Run quick training (few epochs)
4. Run quick evaluation and inspect outputs

Expected outputs are written under `masif_seed_search/data/protein_interaction_nn/`.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys


repo_root = !git rev-parse --show-toplevel
repo_root = Path(repo_root[0])
seed_source = repo_root / "masif_seed_search" / "source"
masif_source = repo_root / "masif" / "source"
protein_interaction_nn_dir = repo_root / "masif_seed_search" / "data" / "protein_interaction_nn"
os.environ["PYTHONPATH"] = f"{seed_source}:{masif_source}:" + os.environ.get("PYTHONPATH", "")

# Add source directories to python path so we can import modules
sys.path.append(str(seed_source))
sys.path.append(str(masif_source))

# work in protein_interaction_nn_dir
os.chdir(protein_interaction_nn_dir)

config_path = protein_interaction_nn_dir / "configs" / "protein_interaction_v1.yaml"
print("Notebook dir:", protein_interaction_nn_dir)
print("Repo root:", repo_root)
print("Config:", config_path)

Notebook dir: /scratch/ymeng/masif-neosurf-af2/masif_seed_search/data/protein_interaction_nn
Repo root: /scratch/ymeng/masif-neosurf-af2
Config: /scratch/ymeng/masif-neosurf-af2/masif_seed_search/data/protein_interaction_nn/configs/protein_interaction_v1.yaml


In [2]:
# Build tiny split CSVs.
cmd = [
    "python",
    str(seed_source / "protein_interaction" / "build_pair_index.py"),
    "--config",
    str(config_path),
    "--tiny",
]
print("Running:")
print(cmd)
subprocess.run(cmd, check=True)


Running:
['python', '/scratch/ymeng/masif-neosurf-af2/masif_seed_search/source/protein_interaction/build_pair_index.py', '--config', '/scratch/ymeng/masif-neosurf-af2/masif_seed_search/data/protein_interaction_nn/configs/protein_interaction_v1.yaml', '--tiny']


CompletedProcess(args=['python', '/scratch/ymeng/masif-neosurf-af2/masif_seed_search/source/protein_interaction/build_pair_index.py', '--config', '/scratch/ymeng/masif-neosurf-af2/masif_seed_search/data/protein_interaction_nn/configs/protein_interaction_v1.yaml', '--tiny'], returncode=0)

In [3]:
pairs_dir = protein_interaction_nn_dir / "splits"
summary_path = pairs_dir / "pairs_summary.json"
print(summary_path.read_text())

{
  "available_ppi_ids": 120,
  "splits": {
    "test": {
      "num_negative": 60,
      "num_positive": 20,
      "num_ppi_ids": 20,
      "num_rows": 80
    },
    "train": {
      "num_negative": 60,
      "num_positive": 20,
      "num_ppi_ids": 20,
      "num_rows": 80
    },
    "val": {
      "num_negative": 30,
      "num_positive": 10,
      "num_ppi_ids": 10,
      "num_rows": 40
    }
  }
}


In [4]:
# Dataset and batch sanity check.
from protein_interaction.utils import load_config
from protein_interaction.dataset import read_pairs_csv, ProteinPairDataset, batch_iterator, sanity_check_batch

cfg = load_config(str(config_path))
pairs_train = pairs_dir / "pairs_train.csv"
records = read_pairs_csv(str(pairs_train))
print("Num train records:", len(records))

ds = ProteinPairDataset(cfg, records, seed=cfg.get("seed", 42))
batch = next(batch_iterator(ds, batch_size=2, descriptor_dim=cfg["model"]["descriptor_dim"], shuffle=False))
sanity_check_batch(batch)

print("query_desc:", batch["query_desc"].shape)
print("query_xyz:", batch["query_xyz"].shape)
print("query_mask:", batch["query_mask"].shape)
print("matched_desc:", batch["matched_desc"].shape)
print("matched_xyz:", batch["matched_xyz"].shape)
print("matched_mask:", batch["matched_mask"].shape)
print("labels:", batch["labels"].reshape(-1))

Num train records: 80
query_desc: (2, 256, 80)
query_xyz: (2, 256, 3)
query_mask: (2, 256)
matched_desc: (2, 1024, 80)
matched_xyz: (2, 1024, 3)
matched_mask: (2, 1024)
labels: [1. 0.]


In [ ]:
# Overfit run: tiny data, many epochs, no early stop.
import tempfile
import yaml

cfg = load_config(str(config_path))
cfg["train"]["epochs"] = 80
cfg["train"]["batch_size"] = 8
cfg["train"]["early_stop_patience"] = 80  # effectively disabled
cfg["data"]["tiny_subset"]["enabled"] = True
cfg["data"]["tiny_subset"]["max_pos_per_split"] = 8  # very small
cfg["data"]["negatives_per_positive"] = 1           # easier memorization

with tempfile.NamedTemporaryFile("w", suffix=".yaml", delete=False) as tmp:
    yaml.safe_dump(cfg, tmp)
    tmp_cfg_path = tmp.name

train_cmd = [
    "python",
    str(seed_source / "protein_interaction" / "train.py"),
    "--config",
    tmp_cfg_path,
]
print("Running:", " ".join(train_cmd))
subprocess.run(train_cmd, check=True)
print("Temporary config:", tmp_cfg_path)

Running: python /scratch/ymeng/masif-neosurf-af2/masif_seed_search/source/protein_interaction/train.py --config /tmp/tmpdji2hjan.yaml


In [10]:
# Evaluate overfit behavior on train + test
for split in ["train", "test"]:
    eval_cmd = [
        "python",
        str(seed_source / "protein_interaction" / "evaluate.py"),
        "--config",
        tmp_cfg_path,
        "--split",
        split,
    ]
    print("Running:", " ".join(eval_cmd))
    subprocess.run(eval_cmd, check=True)

eval_dir = Path(cfg["eval"]["output_dir"])
if not eval_dir.is_absolute():
    eval_dir = repo_root / eval_dir

for split in ["train", "test"]:
    metrics_path = eval_dir / f"metrics_{split}.json"
    print(f"\n== {split.upper()} ==")
    print(metrics_path.read_text())

Running: python /scratch/ymeng/masif-neosurf-af2/masif_seed_search/source/protein_interaction/evaluate.py --config /tmp/tmpto2lm_0t.yaml --split test
Metrics file: /scratch/ymeng/masif-neosurf-af2/masif_seed_search/data/protein_interaction_nn/analysis/v1_eval/metrics_test.json
{
  "loss": 0.6784027457237244,
  "num_samples": 80,
  "pr_auc": 0.2244349513584659,
  "roc_auc": 0.39083333333333337,
  "split": "test"
}
First 10 prediction rows:
query_id,matched_id,label,probability
1A2K_C_AB,1A2K_C_AB,1,0.48334449529647827
1A2K_C_AB,1B2U_A_D,0,0.4841758608818054
1A2K_C_AB,1AGQ_C_D,0,0.48237332701683044
1A2K_C_AB,1ARZ_A_C,0,0.48418131470680237
1A2W_A_B,1A2W_A_B,1,0.4821685254573822
1A2W_A_B,1AGQ_C_D,0,0.4830916225910187
1A2W_A_B,1AN1_E_I,0,0.4831157922744751
1A2W_A_B,1ARZ_A_C,0,0.4823720455169678
1A79_C_B,1A79_C_B,1,0.4821389317512512
1A79_C_B,1B6C_A_B,0,0.4837561249732971
